[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C42_Learning_Theory_Course/03_ntk_infinite_width/03_讲解.ipynb)

# 03 · NTK 与无限宽（计算经验 NTK，核回归预测网络）

目标：从零实现两层 ReLU 网络的**经验 NTK**，看它**随宽度趋稳**（收敛到确定核），并用 **NTK 核回归**预测训练后的网络——把 Jacot 定理变成可对拍的数字。

路线：实现经验 NTK 公式 → 随宽度的相对涨落 ~1/√m → 宽网络 NTK 几乎重合 → NTK 核回归插值 → 核回归 vs 实际训练的宽网络 → lazy(参数微动) → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 纯 numpy；缩放用 NTK 参数化 $1/\sqrt m$（使输出与 NTK 都是 $O(1)$、参数微动 $O(1/\sqrt m)$）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# 两层 ReLU: f(x) = (1/sqrt(m)) sum_j a_j relu(w_j . x)
# 经验 NTK(x,x') = <grad_theta f(x), grad_theta f(x')>，对 a 和 w 的梯度各贡献一项。
def emp_ntk_gram(X, width, seed):
    g = np.random.default_rng(seed); n, d = X.shape
    W = g.standard_normal((width, d))      # w_j ~ N(0, I)
    a = g.standard_normal(width)           # a_j ~ N(0, 1)
    pre = X @ W.T                          # (n, m)
    relu = np.maximum(pre, 0.0)            # (n, m)
    ind  = (pre > 0).astype(float)         # (n, m)
    XXt  = X @ X.T                         # (n, n)
    K1 = (relu @ relu.T) / width                         # ∂a 部分: (1/m) sum relu relu
    K2 = ((ind * (a**2)[None, :]) @ ind.T) * XXt / width  # ∂w 部分: (1/m) sum a_j^2 1·1 (x.x')
    return K1 + K2
print('经验 NTK 公式就绪（两层 ReLU, NTK 参数化 1/sqrt(m)）')

## 1 · 经验 NTK 的对称性与正定性（sanity check）

NTK Gram 矩阵必须是**对称半正定**的（它是特征内积矩阵 $\Phi\Phi^\top$）。先验证这些基本性质。

In [ ]:
n, d = 8, 5
X = rng.standard_normal((n, d)); X /= np.linalg.norm(X, axis=1, keepdims=True)
K = emp_ntk_gram(X, width=512, seed=1)
print('NTK Gram shape:', K.shape)
assert np.allclose(K, K.T, atol=1e-10), 'NTK 必须对称'
evals = np.linalg.eigvalsh(K)
print(f'特征值范围 [{evals.min():.4f}, {evals.max():.4f}]')
assert evals.min() > -1e-9, 'NTK 必须半正定'
print('✅ 经验 NTK 对称半正定 —— 它确实是一个合法的核 Gram 矩阵。')

## 2 · 随宽度稳定：相对涨落 $\sim 1/\sqrt m$

经验 NTK 是 $m$ 个 iid 项的平均，整个矩阵的**相对涨落**应随宽度以 $1/\sqrt m$ 收缩。我们用矩阵层面的稳健度量 $\|\mathrm{std}\|_F/\|\mathrm{mean}\|_F$（Frobenius 相对标准差）——比逐条目 std/mean 稳健（个别条目均值接近 0 会让逐条目比值爆掉）。宽度 16→256（16×），涨落应降约 $\sqrt{16}=4$ 倍。

In [ ]:
def rel_fluctuation(width, n_seeds=40):
    '''整个 NTK 矩阵(去对角)的 Frobenius 相对标准差 = ||std||_F / ||mean||_F。'''
    Ks = np.stack([emp_ntk_gram(X, width, s) for s in range(n_seeds)])
    mean = Ks.mean(0); std = Ks.std(0)
    off = ~np.eye(X.shape[0], dtype=bool)
    return np.linalg.norm(std[off]) / np.linalg.norm(mean[off])
print(f'{"width":>6s}  {"相对涨落(Fro)":>14s}')
rfs = {}
for w in [16, 64, 256, 1024]:
    rfs[w] = rel_fluctuation(w)
    print(f'{w:>6d}  {rfs[w]:>14.5f}')
assert rfs[16] > rfs[64] > rfs[256] > rfs[1024], 'NTK 应随宽度收敛（相对涨落单调减）'
ratio = rfs[16] / rfs[256]
print(f'\n宽度 16->256 (16×) 涨落降 {ratio:.2f}× (理论 sqrt(16)=4)')
assert 2.5 < ratio < 6.0, '相对涨落应以 ~1/sqrt(m) 收缩'
print('✅ NTK 相对涨落 ~ 1/sqrt(width) 收缩 —— 收敛到确定核被钉死。')

## 3 · 宽网络的 NTK 几乎与初始化无关（趋于确定极限核）

若 NTK 收敛到一个确定核，两个**不同随机初始化**的宽网络应给出几乎相同的 NTK 矩阵。验证宽度 4096 时相对差很小。

In [ ]:
KA = emp_ntk_gram(X, 4096, seed=11)
KB = emp_ntk_gram(X, 4096, seed=22)
off = ~np.eye(X.shape[0], dtype=bool)
rel_diff = np.abs(KA - KB)[off].mean() / np.abs(KA)[off].mean()
print(f'两个 width=4096 NTK 矩阵的相对差 = {rel_diff:.4f}')
assert rel_diff < 0.1, '宽网络 NTK 应近似确定（与随机种子无关）'
# 对比：窄网络(width=16)差异大得多
kna = emp_ntk_gram(X, 16, 11); knb = emp_ntk_gram(X, 16, 22)
rd_narrow = np.abs(kna-knb)[off].mean()/np.abs(kna)[off].mean()
print(f'对比 width=16 的相对差 = {rd_narrow:.4f} (宽得多)')
assert rd_narrow > rel_diff, '窄网络 NTK 涨落更大'
print('✅ 宽网络 NTK 趋于一个确定的极限核（不依赖随机初始化）。')

## 4 · NTK 核回归插值训练数据

无限宽训练后预测 $f_\infty(x)=\Theta(x,X)\Theta(X,X)^{-1}y$。在训练点上 ridgeless 解应**精确插值** $y$。

In [ ]:
K = emp_ntk_gram(X, 4096, seed=7)
y = rng.standard_normal(X.shape[0])
alpha = np.linalg.solve(K + 1e-8*np.eye(len(X)), y)   # ridgeless (tiny jitter 数值稳定)
pred_train = K @ alpha
print(f'核回归在训练点的拟合残差 = {np.linalg.norm(pred_train - y):.2e}')
assert np.linalg.norm(pred_train - y) < 1e-4, 'ridgeless NTK 核回归应插值训练数据'
print('✅ NTK 核回归(ridgeless)精确插值训练数据 —— 对应无限宽网络训到零损失。')

## 5 · 核回归 vs 实际训练的宽网络：随宽度趋近

Jacot 定理的核心断言：**实际用 GD 训练的宽网络**的预测 ≈ **NTK 核回归**的预测，且随宽度增大而趋近。
我们实际训练有限宽网络（梯度下降，平方损失），对比其测试点预测与 NTK 核回归预测的差距。

In [ ]:
def train_net(Xtr, ytr, Xte, width, seed, steps=4000, lr=0.2):
    '''实际训练两层 ReLU(NTK 参数化), 返回测试点预测。'''
    g = np.random.default_rng(seed); n, d = Xtr.shape
    W = g.standard_normal((width, d)); a = g.standard_normal(width)
    s = 1.0/np.sqrt(width)
    def fwd(Xb, W, a):
        return s * np.maximum(Xb @ W.T, 0.0) @ a
    for _ in range(steps):
        pre = Xtr @ W.T; relu = np.maximum(pre, 0.0); ind = (pre > 0).astype(float)
        r = s * (relu @ a) - ytr                       # 残差 (n,)
        ga = s * (relu.T @ r) / n                       # dL/da
        gW = s * ((ind * (r[:,None]*a[None,:])).T @ Xtr) / n  # dL/dW
        a = a - lr * ga; W = W - lr * gW
    return fwd(Xte, W, a)
def ntk_predict(Xtr, ytr, Xte, width, seed):
    '''NTK 核回归预测(同一初始化的经验 NTK)。'''
    Xall = np.vstack([Xtr, Xte]); Kall = emp_ntk_gram(Xall, width, seed)
    n = len(Xtr); Ktr = Kall[:n,:n]; Kte = Kall[n:,:n]
    alpha = np.linalg.solve(Ktr + 1e-6*np.eye(n), ytr)
    return Kte @ alpha
ntr, nte, d = 12, 6, 4
Xtr = rng.standard_normal((ntr,d)); Xtr/=np.linalg.norm(Xtr,axis=1,keepdims=True)
Xte = rng.standard_normal((nte,d)); Xte/=np.linalg.norm(Xte,axis=1,keepdims=True)
ytr = rng.standard_normal(ntr)
print(f'{"width":>6s}  {"||训练网络 - NTK核回归||":>26s}')
prev = None
for w in [50, 200, 1000]:
    p_net = train_net(Xtr, ytr, Xte, w, seed=5)
    p_ntk = ntk_predict(Xtr, ytr, Xte, w, seed=5)
    diff = np.linalg.norm(p_net - p_ntk)
    print(f'{w:>6d}  {diff:>26.4f}')
    prev = diff
# 宽度足够大时两者应接近
p_net = train_net(Xtr, ytr, Xte, 1000, seed=5); p_ntk = ntk_predict(Xtr, ytr, Xte, 1000, seed=5)
assert np.linalg.norm(p_net - p_ntk) < 0.5*np.linalg.norm(p_ntk) + 0.3, '宽网络预测应接近 NTK 核回归'
print('✅ 实际训练的宽网络预测趋近 NTK 核回归预测 —— Jacot 等价的数值确认。')

## 6 · lazy training：宽网络参数只微动 $O(1/\sqrt m)$

惰性训练的标志：宽网络训练后参数相对初始化的移动 $\|\theta_T-\theta_0\|/\|\theta_0\|$ 随宽度**减小**（$\sim1/\sqrt m$）。

In [ ]:
def param_move(width, seed=5, steps=2000, lr=0.2):
    g = np.random.default_rng(seed); n, d = Xtr.shape
    W0 = g.standard_normal((width,d)); a0 = g.standard_normal(width)
    W = W0.copy(); a = a0.copy(); s = 1.0/np.sqrt(width)
    for _ in range(steps):
        pre = Xtr@W.T; relu = np.maximum(pre,0.0); ind=(pre>0).astype(float)
        r = s*(relu@a) - ytr
        a = a - lr*(s*(relu.T@r)/n)
        W = W - lr*(s*((ind*(r[:,None]*a[None,:])).T@Xtr)/n)
    move = np.sqrt(np.sum((W-W0)**2)+np.sum((a-a0)**2))
    init = np.sqrt(np.sum(W0**2)+np.sum(a0**2))
    return move/init
print(f'{"width":>6s}  {"相对参数移动":>14s}')
moves = {}
for w in [50, 200, 800]:
    moves[w] = param_move(w)
    print(f'{w:>6d}  {moves[w]:>14.5f}')
assert moves[50] > moves[200] > moves[800], '越宽参数移动越小(惰性)'
print('✅ 越宽，参数相对移动越小 —— lazy training: 网络停在初始化附近。')

---
## ✏️ 练习 1：NTK 矩阵的特征值谱

NTK 谱决定可学习性（大特征值方向学得快）。实现 `ntk_spectrum(X, width, seed)` 返回 NTK Gram 矩阵的特征值（降序）。并验证：谱非负、最大特征值随训练集相关性结构变化。

In [ ]:
def ntk_spectrum(X, width, seed):
    # TODO: 算 emp_ntk_gram(X, width, seed) 的特征值, 降序返回 (用 np.linalg.eigvalsh)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Xs = rng.standard_normal((10, 5)); Xs /= np.linalg.norm(Xs, axis=1, keepdims=True)
spec = ntk_spectrum(Xs, 1024, 0)
assert len(spec) == 10 and np.all(spec >= -1e-9), '谱应非负'
assert np.all(np.diff(spec) <= 1e-9), '应降序'
print(f'NTK 谱(前5): {np.round(spec[:5], 4)}')
print(f'谱的条件数 lambda_max/lambda_min = {spec[0]/spec[-1]:.1f}')
print('✅ 练习 1 通过：NTK 谱非负降序；其条件数影响核回归/梯度流的收敛速度')

## ✏️ 练习 2：宽度趋势 —— 相对涨落随宽度递减

实现 `fluctuation_trend(widths)` 返回每个宽度的相对涨落列表（复用 worked 2 的 `rel_fluctuation`）。验证列表单调递减。

In [ ]:
def fluctuation_trend(widths, n_seeds=30):
    # TODO: 对每个 width 调用 rel_fluctuation(width, n_seeds), 返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
trend = fluctuation_trend([32, 128, 512])
print('相对涨落:', [f'{x:.4f}' for x in trend])
assert trend[0] > trend[1] > trend[2], '相对涨落应随宽度单调递减'
print('✅ 练习 2 通过：宽度越大 NTK 越稳定（涨落递减）')

## ✏️ 练习 3：lazy vs feature learning —— 参数移动随宽度

实现 `move_trend(widths)` 返回每个宽度训练后的相对参数移动（复用 worked 6 的 `param_move`）。验证：宽度越大移动越小（惰性越强）。

In [ ]:
def move_trend(widths):
    # TODO: 对每个 width 调用 param_move(width), 返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
mt = move_trend([50, 200, 800])
print('相对参数移动:', [f'{x:.4f}' for x in mt])
assert mt[0] > mt[1] > mt[2], '宽度越大参数移动越小(越惰性)'
print('✅ 练习 3 通过：宽 -> 惰性(NTK区, 固定特征); 窄 -> 参数大动(特征学习)')

## ✏️ 练习 4：NTK 核回归预测

实现 `ntk_regress(Xtr, ytr, Xte, width, seed, ridge)` 返回测试点的 NTK 核回归预测 $\Theta(X_{te},X_{tr})(\Theta(X_{tr},X_{tr})+\lambda I)^{-1}y$。

In [ ]:
def ntk_regress(Xtr, ytr, Xte, width, seed, ridge=1e-6):
    # TODO: 拼接 Xtr,Xte 算大 NTK -> 取 Ktr=K[:n,:n], Kte=K[n:,:n]
    #       alpha = solve(Ktr + ridge*I, ytr); 返回 Kte @ alpha
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
pte = ntk_regress(Xtr, ytr, Xte, 1024, 3)
assert pte.shape == (len(Xte),), '预测形状应为测试点数'
# 训练点上(ridge->0)应近似插值
ptr = ntk_regress(Xtr, ytr, Xtr, 1024, 3, ridge=1e-8)
assert np.linalg.norm(ptr - ytr) < 1e-2, 'ridgeless 在训练点应近似插值'
print(f'测试点 NTK 预测: {np.round(pte, 3)}')
print('✅ 练习 4 通过：NTK 核回归可预测任意测试点，训练点处插值')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1
def ntk_spectrum(X, width, seed):
    return np.linalg.eigvalsh(emp_ntk_gram(X, width, seed))[::-1]

In [ ]:
# 练习 2
def fluctuation_trend(widths, n_seeds=30):
    return [rel_fluctuation(w, n_seeds) for w in widths]

In [ ]:
# 练习 3
def move_trend(widths):
    return [param_move(w) for w in widths]

In [ ]:
# 练习 4
def ntk_regress(Xtr, ytr, Xte, width, seed, ridge=1e-6):
    Xall = np.vstack([Xtr, Xte]); K = emp_ntk_gram(Xall, width, seed)
    n = len(Xtr); Ktr = K[:n,:n]; Kte = K[n:,:n]
    alpha = np.linalg.solve(Ktr + ridge*np.eye(n), ytr)
    return Kte @ alpha

---
## 🧪 真实数据胶囊：NTK 核回归在真实数据(digits 子集)上分类

用真实数据（sklearn digits 的两类子集；失败回退到真实形状的合成数据）做 NTK 核回归二分类，验证它能学到非平凡的测试准确率——无限宽网络作为核方法确实能分类真实数据。

In [ ]:
try:
    from sklearn.datasets import load_digits
    dig = load_digits(); m = np.isin(dig.target, [0, 1])
    Xr = dig.data[m].astype(float); yr = (dig.target[m]*2 - 1).astype(float)
    Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-9)
    src = 'sklearn digits 0vs1 (真实)'
except Exception as e:
    g = np.random.default_rng(0); nT = 360
    Xr = g.standard_normal((nT, 64)); yr = np.sign(g.standard_normal(nT))
    Xr /= np.linalg.norm(Xr, axis=1, keepdims=True); src = f'回退合成: {type(e).__name__}'
ntr = int(0.7*len(Xr)); idx = np.random.default_rng(0).permutation(len(Xr))
tr, te = idx[:ntr], idx[ntr:]
print(f'数据来源: {src}; 共 {len(Xr)} 样本, 训练 {len(tr)} 测试 {len(te)}')

In [ ]:
Ktr = emp_ntk_gram(Xr[tr], 2048, seed=0)
Kall = emp_ntk_gram(np.vstack([Xr[tr], Xr[te]]), 2048, seed=0)
n = len(tr); Ktr = Kall[:n,:n]; Kte = Kall[n:,:n]
alpha = np.linalg.solve(Ktr + 1e-4*np.eye(n), yr[tr])
pred = np.sign(Kte @ alpha)
acc = np.mean(pred == yr[te])
print(f'NTK 核回归测试准确率 = {acc:.3f}')
assert acc > 0.8, 'NTK 核回归应能在真实(可分)数据上学到非平凡准确率'
print('✅ 无限宽网络(NTK 核方法)在真实数据上分类成功')

**🧪 胶囊练习**：实现 `ntk_train_acc()` 返回 NTK 核回归在**训练集**上的准确率（ridgeless 应接近 1，因为插值）。

In [ ]:
def ntk_train_acc():
    # TODO: 用上面的 Ktr, alpha, 在训练点预测 sign(Ktr @ alpha), 返回与 yr[tr] 的符合率
    raise NotImplementedError

In [ ]:
# 胶囊自测
def ntk_train_acc():
    return np.mean(np.sign(Ktr @ alpha) == yr[tr])
ta = ntk_train_acc()
print(f'NTK 核回归训练准确率 = {ta:.3f}')
assert ta > 0.98, 'ridgeless 核回归在训练集上应近乎插值(准确率~1)'
print('✅ 胶囊练习通过：训练集上插值(准确率~1)，对应无限宽网络训到零损失')

In [ ]:
# 📖 胶囊参考答案
def ntk_train_acc():
    return np.mean(np.sign(Ktr @ alpha) == yr[tr])

---
### 小结
- NTK $\Theta(x,x')=\langle\nabla_\theta f(x),\nabla_\theta f(x')\rangle$；平方损失下输出残差按 $\Theta$ 线性演化。
- 经验 NTK 随宽度**收敛到确定核**：相对涨落 ~$1/\sqrt m$（已对拍），宽网络 NTK 与种子无关。
- 无限宽训练 = NTK 核回归 $f_\infty=\Theta(x,X)\Theta(X,X)^{-1}y$，ridgeless 在训练点插值（已验证）。
- 实际训练的宽网络预测趋近 NTK 核回归（Jacot 等价的数值确认）；参数微动 ~$1/\sqrt m$（lazy）。
- 诚实差距：NTK 在 lazy 区固定特征，真实网络会**特征学习**、常超过对应 NTK。

下一站：**模块 04 · 隐式偏置**——既然有无穷多零损失解，GD 究竟挑了哪个？为什么它泛化好？